# Two Agents Talking

This is the smallest useful multi-agent pattern in NOOA: create two agent objects, call one, and pass its return value into the other.

## Setup

The quickstart helper picks an LLM from your environment. Set one of `OPENAI_API_KEY`, `NVIDIA_API_KEY`, or `NVIDIA_INFERENCE_API_KEY` before running the notebook.

In [4]:
from nooa import Agent

from nooa.unifiedllm.registry import get_llm_client

# model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")                                        # Anthropic
# model = get_llm_client("gpt-5-mini", api_key="your-api-key")                                              # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")                       # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1")                # vLLM (local, no key)
model = get_llm_client("openai/openai/openai/gpt-5.5", api_key="sk-NghOY6FpXTh5sK4Y5fquog", api_base="https://inference-api.nvidia.com/v1")  # NVIDIA hosted (OpenAI-compatible)


## One Tiny Chat Agent

Both participants use the same class. Different constructor state gives each instance a different voice.

In [9]:
class ChatAgent(Agent, llm=model):
    """A tiny chat participant."""
    async def say(self, incoming_message: str) -> str:
        """Write the next chat message"""
        ...


class Alice(ChatAgent):
    """A tiny chat participant."""


class Bob(ChatAgent):
    """A tiny chat participant."""


## Let Them Talk

There is no special multi-agent API here. `await alice.say(...)` returns a Python string; `bob.say(...)` receives that string like any other argument.

In [10]:
alice = Alice()
bob = Bob()

bob_answer = "Ask Bob what makes a NOOA agent an ordinary Python object."
while True:
    alice_answer = await alice.say(bob_answer)
    print(alice_answer)
    bob_answer = await bob.say(alice_answer)
    print(bob_answer)
    input('proceed?')


Bob, what makes a NOOA agent an ordinary Python object?
A NOOA agent is an ordinary Python object because it’s just an instance of a normal Python class—you can construct it, pass it around, store state on it, and call its async methods like any other object. The “agent” behavior comes from how those methods are executed, not from it being some special separate runtime entity.
That makes sense: the agent is just a regular class instance, while NOOA supplies the execution strategy around its async methods.
Exactly. You write normal Python classes and async methods, and NOOA wraps the method call with the chosen strategy—so the object model stays familiar while the execution can involve LLM reasoning, code, tools, or delegation.
Nice—so NOOA keeps the developer ergonomics of plain Python while letting each method choose a richer execution mode.


CancelledError: 

## Takeaway

Two agents talk by exchanging ordinary Python values. If you need more structure later, change the return type from `str` to a Pydantic model or move this sequencing into a pure-Python orchestrator method.